In [ ]:
# 1) Cargar la base de datos (sube el archivo Datos_históricos_RKLB.csv)
from google.colab import files
import io, numpy as np, pandas as pd, matplotlib.pyplot as plt

subido = files.upload()
nombre = list(subido.keys())[0]
df = pd.read_csv(io.BytesIO(subido[nombre]))

# Limpieza: fechas dd.mm.yyyy, números y volumen ("23.31M")
df["Fecha"] = pd.to_datetime(df["Fecha"], format="%d.%m.%Y")
df["Vol."] = df["Vol."].str.replace("M", "", regex=False).astype(float) * 1e6
df["% var."] = df["% var."].str.replace("%", "", regex=False).astype(float)
df = df.sort_values("Fecha").reset_index(drop=True)

print(f"{len(df)} meses: {df['Fecha'].min().date()} a {df['Fecha'].max().date()}")
df.tail()

In [ ]:
# 2) Serie de tiempo mensual del cierre
serie = df.set_index("Fecha")["Cierre"]
serie.index = serie.index.to_period("M").to_timestamp()
serie = serie.asfreq("MS")

plt.figure(figsize=(11, 4.5))
plt.plot(serie, color="tab:blue", linewidth=2)
plt.yscale("log")
plt.title("RKLB - Cierre mensual (escala logarítmica)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 3) Modelos de pronóstico hasta octubre 2027
from statsmodels.tsa.holtwinters import ExponentialSmoothing

y_log = np.log(serie.values)
t = np.arange(len(y_log))
h = 12                                   # nov-2026 ... oct-2027
fechas_fut = pd.date_range(serie.index[-1] + pd.offsets.MonthBegin(1), periods=h, freq="MS")
t_fut = np.arange(len(y_log), len(y_log) + h)

# Modelo A: tendencia lineal sobre log(precio) por mínimos cuadrados
m, b = np.polyfit(t, y_log, 1)
pred_tendencia = np.exp(m * t_fut + b)
print(f"Tendencia log-lineal: crecimiento mensual = {np.exp(m)-1:.2%}")

# Modelo B: suavizamiento exponencial de Holt con tendencia amortiguada
holt = ExponentialSmoothing(y_log, trend="add", damped_trend=True).fit()
pred_holt = np.exp(holt.forecast(h))

# Modelo C: simulación Monte Carlo (bootstrap de rendimientos logarítmicos mensuales)
rng = np.random.default_rng(42)
rend = np.diff(y_log)
n_sim = 10000
sim = rng.choice(rend - rend.mean(), size=(n_sim, h))   # sin deriva histórica (conservador)
trayectorias = serie.values[-1] * np.exp(np.cumsum(sim, axis=1))
p10, p50, p90 = np.percentile(trayectorias, [10, 50, 90], axis=0)

res = pd.DataFrame({
    "Tendencia log-lineal": pred_tendencia,
    "Holt amortiguado": pred_holt,
    "MC mediana": p50,
    "MC P10": p10,
    "MC P90": p90
}, index=fechas_fut).round(2)
res

In [ ]:
# 4) Gráfica: historia + pronóstico a octubre 2027
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(serie, color="tab:blue", linewidth=2.5, label="Histórico")
ax.plot(fechas_fut, pred_tendencia, "--", color="tab:orange", linewidth=2, label="Tendencia log-lineal")
ax.plot(fechas_fut, pred_holt, "--", color="tab:green", linewidth=2, label="Holt amortiguado")
ax.plot(fechas_fut, p50, color="tab:red", linewidth=2.5, label="Monte Carlo (mediana)")
ax.fill_between(fechas_fut, p10, p90, color="tab:red", alpha=0.15, label="Rango 80% (P10-P90)")
ax.axvline(serie.index[-1], color="gray", linestyle=":")
ax.set_yscale("log")
ax.set_title("RKLB: pronóstico del cierre mensual hasta octubre 2027", fontsize=15, fontweight="bold")
ax.set_ylabel("Precio de cierre (USD, escala log)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

ult = serie.iloc[-1]
print(f"Último cierre ({serie.index[-1].strftime('%b-%Y')}): {ult:.2f}")
print(f"Octubre 2027 -> Tendencia: {pred_tendencia[-1]:.2f} | Holt: {pred_holt[-1]:.2f} | "
      f"MC mediana: {p50[-1]:.2f} (rango 80%: {p10[-1]:.2f} a {p90[-1]:.2f})")